# Data Pipeline

In [ ]:
from pathlib import Path
from torch.utils.data import Dataset
from torchvision.datasets import FGVCAircraft

# Notebooks don't share variables, so we re-declare our chosen classes here.
MY_CLASSES = ['Boeing 737', 'A320', 'A340', 'Boeing 777', 'A380']

DATA_DIR = (Path('..') / 'data').resolve()

# Load the FULL train split (all 70 classes). It's already downloaded from
# notebook 01, so this is instant. We'll filter it in the next cell.
full_train = FGVCAircraft(root=str(DATA_DIR), split='train',
                          annotation_level='family', download=False)

print('Full train set:', len(full_train), 'images,', len(full_train.classes), 'classes')

## Filter dataset to selected classes

In [ ]:
class FilteredAircraft(Dataset):
    """Wraps an FGVCAircraft dataset, keeping only `class_names` and
    renumbering their labels to 0..len(class_names)-1."""

    def __init__(self, base_ds, class_names):
        self.base = base_ds
        self.classes = class_names  # our 5 names, in order -> new labels 0..4

        # Original label-number for each of our chosen class names.
        # e.g. 'A320' -> 2, 'Boeing 737' -> 15, ...
        keep_ids = [base_ds.classes.index(name) for name in class_names]

        # Map: original label-number -> new label 0..4 (Problem 2: remapping).
        self.old_to_new = {old: new for new, old in enumerate(keep_ids)}

        # Positions in the base dataset whose label we want to keep (Problem 1).
        # base_ds._labels is the list of original label-numbers, one per image.
        self.indices = [i for i, lab in enumerate(base_ds._labels)
                        if lab in self.old_to_new]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        # Fetch the original (image, original_label) ...
        img, old_label = self.base[self.indices[i]]
        # ... and return it with the REMAPPED label.
        return img, self.old_to_new[old_label]


train_ds = FilteredAircraft(full_train, MY_CLASSES)

print('Filtered train set:', len(train_ds), 'images')
print('Label mapping:')
for name, new_id in zip(train_ds.classes, range(len(train_ds.classes))):
    print(f'  {new_id} -> {name}')

# Quick peek: one sample should be (PIL image, an int in 0..4)
img, label = train_ds[0]
print('\nExample sample -> image:', img.size, '| label:', label, '=', train_ds.classes[label])

## Use official train/val/test split ny FGVC's

In [ ]:
# Load the val and test splits (already downloaded) and filter them with the
# SAME wrapper + SAME class list -> labels stay consistent (0=Boeing 737, etc.).
full_val  = FGVCAircraft(root=str(DATA_DIR), split='val',  annotation_level='family', download=False)
full_test = FGVCAircraft(root=str(DATA_DIR), split='test', annotation_level='family', download=False)

val_ds  = FilteredAircraft(full_val,  MY_CLASSES)
test_ds = FilteredAircraft(full_test, MY_CLASSES)

print(f'train: {len(train_ds):4d} images')
print(f'val:   {len(val_ds):4d} images')
print(f'test:  {len(test_ds):4d} images')

# Sanity: the three splits must not overlap (no leakage). FGVC guarantees this,
# but let's confirm by checking the underlying image file paths are disjoint.
train_files = {full_train._image_files[i] for i in train_ds.indices}
val_files   = {full_val._image_files[i]   for i in val_ds.indices}
test_files  = {full_test._image_files[i]  for i in test_ds.indices}
print('\nOverlap train/val :', len(train_files & val_files))
print('Overlap train/test:', len(train_files & test_files))
print('Overlap val/test  :', len(val_files & test_files))

## Define transforms and add aufmentations

In [ ]:
from torchvision import transforms

# Pretrained ImageNet models expect 224x224 inputs normalized with these stats.
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
BANNER_PX = 20  # FGVC images have a 20px copyright banner along the bottom


def remove_banner(img):
    """Crop off the bottom 20px copyright banner. FGVC's docs require this so the
    model learns aircraft, not photographer/copyright text. Applied FIRST, before
    any resize/crop, using each image's actual height."""
    w, h = img.size
    return img.crop((0, 0, w, h - BANNER_PX))


# TRAIN: remove banner -> random crop + flip (augmentation) -> tensor -> normalize.
train_tf = transforms.Compose([
    transforms.Lambda(remove_banner),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# VAL / TEST: remove banner -> deterministic resize + center crop -> tensor -> normalize.
eval_tf = transforms.Compose([
    transforms.Lambda(remove_banner),
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Attach each recipe to the matching split's base dataset. FGVCAircraft applies
# its .transform on access, and our FilteredAircraft passes images through it.
full_train.transform = train_tf
full_val.transform   = eval_tf
full_test.transform  = eval_tf

# Check: a sample is now a TENSOR of shape [3, 224, 224] (channels, H, W).
img, label = train_ds[0]
print('Sample type :', type(img).__name__)
print('Sample shape:', tuple(img.shape), '(channels, height, width)')
print('Value range :', f'{img.min():.2f} to {img.max():.2f}  (normalized, so not 0-1)')
print('Label       :', label, '=', train_ds.classes[label])

## Wrap in DataLoaders

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32  # how many images the model sees at once (fits easily in 8 GB)

# shuffle=True for train: the model shouldn't learn the order of images.
# shuffle=False for val/test: order doesn't matter and stable order is tidier.
# num_workers=0: on Windows + Jupyter, parallel workers often hang, so we keep
#   it simple here. (In a .py script later we can bump this up for speed.)
# pin_memory=True: slightly faster CPU->GPU transfer.
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=True)

print('Batches per epoch ->',
      'train:', len(train_loader),
      '| val:', len(val_loader),
      '| test:', len(test_loader))

## Check

In [ ]:
# Pull ONE batch from the train loader and inspect it.
images, labels = next(iter(train_loader))

print('Batch of images:', tuple(images.shape), '-> (batch, channels, height, width)')
print('Batch of labels:', tuple(labels.shape))
print('Label values   :', labels.tolist())
print('\nIf images is (32, 3, 224, 224) and labels are 32 ints in 0-4, the pipeline works!')